Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2


Edge Detection


In [ ]:
image_path = r'D:\PythonProject1\Intern\Images\img111.png'
image = cv2.imread(image_path)
gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
blurred = cv2.GaussianBlur(gray_image, (3,3), 0)
_, thresh = cv2.threshold(gray_image, 240, 245, cv2.THRESH_BINARY_INV)
canny = cv2.Canny(thresh,50,150)
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.show()
plt.imshow(canny, cmap='gray')
plt.show()

Seperate Each Torn Fragment using Contours

In [ ]:
_, thresh = cv2.threshold(blurred, 200, 255, cv2.THRESH_BINARY) 

kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
closed = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
contours, hierarchy = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
min_area = 500 
valid_contours = [c for c in contours if cv2.contourArea(c) > min_area]
contour_img = image.copy()
cv2.drawContours(contour_img, valid_contours, -1, (0, 255, 0), 3)


In [ ]:
isolated_pieces_list = []  
count=0
for contour in valid_contours: 
    x, y, w, h = cv2.boundingRect(contour)
    pad =0
    
    x_start, x_end = max(0, x - pad), min(image.shape[1], x + w + pad)
    y_start, y_end = max(0, y - pad), min(image.shape[0], y + h + pad)
    
    cropped_piece = image[y_start:y_end, x_start:x_end]
    
    mask = np.zeros_like(closed)
    cv2.drawContours(mask, [contour], -1, 255, -1) 

    cropped_mask = mask[y_start:y_end, x_start:x_end]
    
    isolated_piece = cv2.bitwise_and(cropped_piece, cropped_piece, mask=cropped_mask)
    
    rgb_piece = cv2.cvtColor(isolated_piece, cv2.COLOR_BGR2RGB)
    isolated_pieces_list.append(rgb_piece)
    count += 1
    # plt.imshow(rgb_piece)
    # plt.show()

In [ ]:
img = isolated_pieces_list[0]

gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
_, thresh = cv2.threshold(gray, 15, 255, cv2.THRESH_BINARY)
y, x = np.where(thresh == 255)
highest = np.argmax(x)
peak_x = x[highest]
peak_y = y[highest]
output_img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR) if len(img.shape) == 2 else img.copy()
cv2.circle(output_img, (peak_x, peak_y), 0, (0,0,255), -1)
# plt.imshow(cv2.cvtColor(output_img, cv2.COLOR_BGR2RGB))
# plt.show()
# print(peak_y)
# print(peak_x)

Add padding using peak

In [ ]:
bottom = np.max(y)  
right = np.max(x)  
top= np.min(y) 
left = np.min(x)  
padding_color=[0,0,0]
output= cv2.cvtColor(img, cv2.COLOR_GRAY2BGR) if len(img.shape) == 2 else img.copy()
output = cv2.copyMakeBorder(output,10,10,10,10,borderType=cv2.BORDER_CONSTANT,value=padding_color)

bottom += 10
right += 10
top += 10
left += 10
y1 = x[y == (bottom-10)]+10
for peak_y in y1:
    cv2.circle(output, (peak_y, bottom), 0, (0, 0, 255), -1)
    # print(peak_y, bottom)

y2 = y[x == right-10]+10 
for peak_y in y2:
    cv2.circle(output, (right, peak_y), 0, (0, 0, 255), -1) 
    #print(x2,peak_y)

y3 = x[y == top-10]+10 
for peak_y in y3:
    cv2.circle(output, (peak_y, top), 0, (0, 0, 255), -1)
    #print(peak_y, x3)
y4 = y[x == left-10]+10 
for peak_y in y4:
    cv2.circle(output, (left, peak_y), 0, (0, 0, 255), -1)
plt.imshow(cv2.cvtColor(output, cv2.COLOR_BGR2RGB))
plt.show()



In [ ]:
for i in isolated_pieces_list:
    padded_image = cv2.copyMakeBorder(i, 10, 10, 10, 10, borderType=cv2.BORDER_CONSTANT, value=[0,0,0])
    gray = cv2.cvtColor(padded_image, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 15, 255, cv2.THRESH_BINARY)
    points = cv2.findNonZero(thresh)
    x, y, w, h = cv2.boundingRect(points)

    tol = 1.5 

    top_deviations = []
    bottom_deviations = []

    for col in range(x, x + w):
        white_pixels = np.where(thresh[:, col] == 255)[0]
        if len(white_pixels) > 0:
            actual_top_edge = white_pixels[0]
            top_deviations.append(int(actual_top_edge - y))
                    
            actual_bottom_edge = white_pixels[-1]
            absolute_bottom_peak = y + h
            bottom_deviations.append(int(absolute_bottom_peak - actual_bottom_edge))

    std_top = np.std(top_deviations) if top_deviations else 0
    std_bottom = np.std(bottom_deviations) if bottom_deviations else 0

    top1_is_torn = std_top > tol
    bottom1_is_torn = std_bottom > tol

    print(f"Top :    {'TORN' if top1_is_torn else 'STRAIGHT'} (avg: {std_top:.2f}px)")
    print(f"Bottom : {'TORN' if bottom1_is_torn else 'STRAIGHT'} (avg: {std_bottom:.2f}px)")


    left_avg = []
    right_avg = []

    for row in range(y, y + h):
        white_pixels = np.where(thresh[row, :] == 255)[0]
        if len(white_pixels) > 0:
            actual_left_edge = white_pixels[0]
            left_avg.append(int(actual_left_edge - x))
                    
            actual_right_edge = white_pixels[-1]
            absolute_right_peak = x + w
            right_avg.append(int(absolute_right_peak - actual_right_edge))

    std_left = np.std(left_avg) if left_avg else 0
    std_right = np.std(right_avg) if right_avg else 0

    left1_is_torn = std_left > tol
    right1_is_torn = std_right > tol

    print(f"Left :   {'TORN' if left1_is_torn else 'STRAIGHT'} (avg: {std_left:.2f}px)")
    print(f"Right :  {'TORN' if right1_is_torn else 'STRAIGHT'} (avg: {std_right:.2f}px)")

    plt.imshow(thresh, cmap='gray')
    plt.show()

In [ ]:
img=isolated_pieces_list[0]
img = cv2.copyMakeBorder(img,10,10,10,10,borderType=cv2.BORDER_CONSTANT,value=padding_color)
_, img = cv2.threshold(img, 15, 255, cv2.THRESH_BINARY)
a1=cv2.Canny(img,150,150)
y_arr, x_arr = np.where(a1 == 255)

left = np.min(x_arr)
right = np.max(x_arr)
b11= []
for col_x in range(left, right + 1):
    pixels_in_col = y_arr[x_arr == col_x]
    
    if len(pixels_in_col) > 0:
        top_y = np.min(pixels_in_col)
        
        b11.append((int(col_x), int(top_y)))
plt.imshow(cv2.cvtColor(a1, cv2.COLOR_BGR2RGB))
plt.show()
a2 = cv2.copyMakeBorder(thresh,10,10,10,10,borderType=cv2.BORDER_CONSTANT,value=padding_color)
g1= 5
b1=(left , top - g1)
b2=(right , bottom + g1)

cv2.rectangle(a2, b1, b2, (255, 255, 255), 1)
y_arr, x_arr = np.where(a2 == 255)

left = np.min(x_arr)
right = np.max(x_arr)
b22 = []
for col_x in range(left, right + 1):
    pixels_in_col = y_arr[x_arr == col_x]
    
    if len(pixels_in_col) > 0:
        top_y = np.min(pixels_in_col)
        
        b22.append((int(col_x), int(top_y)))
distances = [int(y11) - int(y22) for (_, y11), (_, y22) in zip(b11, b22)]

print(b11)
print(b22)
print(distances)
# plt.imshow(cv2.cvtColor(a2, cv2.COLOR_BGR2RGB))
# plt.show()


In [ ]:
for i in isolated_pieces_list:
    padded_image = cv2.copyMakeBorder(i, 10, 10, 10, 10, borderType=cv2.BORDER_CONSTANT, value=[0,0,0])
    gray = cv2.cvtColor(padded_image, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 15, 255, cv2.THRESH_BINARY)
    points = cv2.findNonZero(thresh)
    x, y, w, h = cv2.boundingRect(points)

    tol = 1.5 

    top_deviations = []
    bottom_deviations = []

    for col in range(x, x + w):
        white_pixels = np.where(thresh[:, col] == 255)[0]
        if len(white_pixels) > 0:
            actual_top_edge = white_pixels[0]
            top_deviations.append(int(actual_top_edge - y))
                    
            actual_bottom_edge = white_pixels[-1]
            absolute_bottom_peak = y + h
            bottom_deviations.append(int(absolute_bottom_peak - actual_bottom_edge))

    std_top = np.std(top_deviations) if top_deviations else 0
    std_bottom = np.std(bottom_deviations) if bottom_deviations else 0

    top1_is_torn = std_top > tol
    bottom1_is_torn = std_bottom > tol

    print(f"Top :    {'TORN' if top1_is_torn else 'STRAIGHT'} (avg: {std_top:.2f}px)")
    print(f"Bottom : {'TORN' if bottom1_is_torn else 'STRAIGHT'} (avg: {std_bottom:.2f}px)")


    left_avg = []
    right_avg = []

    for row in range(y, y + h):
        white_pixels = np.where(thresh[row, :] == 255)[0]
        if len(white_pixels) > 0:
            actual_left_edge = white_pixels[0]
            left_avg.append(int(actual_left_edge - x))
                    
            actual_right_edge = white_pixels[-1]
            absolute_right_peak = x + w
            right_avg.append(int(absolute_right_peak - actual_right_edge))

    std_left = np.std(left_avg) if left_avg else 0
    std_right = np.std(right_avg) if right_avg else 0

    left1_is_torn = std_left > tol
    right1_is_torn = std_right > tol

    print(f"Left :   {'TORN' if left1_is_torn else 'STRAIGHT'} (avg: {std_left:.2f}px)")
    print(f"Right :  {'TORN' if right1_is_torn else 'STRAIGHT'} (avg: {std_right:.2f}px)")

    plt.imshow(thresh, cmap='gray')
    plt.show()

In [ ]:
topp,bottomm,rightt,leftt=[],[],[],[]

In [ ]:
image_path = r'D:\PythonProject1\Intern\Images\img111.png'
image = cv2.imread(image_path)

gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
blurred = cv2.GaussianBlur(gray_image, (3,3), 0)

_, thresh_init = cv2.threshold(blurred, 200, 255, cv2.THRESH_BINARY) 

kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
closed = cv2.morphologyEx(thresh_init, cv2.MORPH_CLOSE, kernel)
contours, hierarchy = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

min_area = 500 
valid_contours = [c for c in contours if cv2.contourArea(c) > min_area]

isolated_pieces_list = []  
for contour in valid_contours: 
    x, y, w, h = cv2.boundingRect(contour)
    pad = 0
    
    x_start, x_end = max(0, x - pad), min(image.shape[1], x + w + pad)
    y_start, y_end = max(0, y - pad), min(image.shape[0], y + h + pad)
    
    cropped_piece = image[y_start:y_end, x_start:x_end]
    
    mask = np.zeros_like(closed)
    cv2.drawContours(mask, [contour], -1, 255, -1) 

    cropped_mask = mask[y_start:y_end, x_start:x_end]
    isolated_piece = cv2.bitwise_and(cropped_piece, cropped_piece, mask=cropped_mask)
    
    isolated_pieces_list.append(isolated_piece)

for  current_piece in isolated_pieces_list:
    padded_image = cv2.copyMakeBorder(current_piece, 10, 10, 10, 10, borderType=cv2.BORDER_CONSTANT, value=[0,0,0])
    
    gray = cv2.cvtColor(padded_image, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 15, 255, cv2.THRESH_BINARY)
    
    points = cv2.findNonZero(thresh)
    
    x_box, y_box, w_box, h_box = cv2.boundingRect(points)

    tol = 1.5 
    top_deviations, bottom_deviations = [], []

    for col in range(x_box, x_box + w_box):
        white_pixels = np.where(thresh[:, col] == 255)[0]
        if len(white_pixels) > 0:
            top_deviations.append(int(white_pixels[0] - y_box))
            bottom_deviations.append(int((y_box + h_box) - white_pixels[-1]))

    std_top = np.std(top_deviations) if top_deviations else 0
    std_bottom = np.std(bottom_deviations) if bottom_deviations else 0
    
    top_is_torn = std_top > tol
    bottom_is_torn = std_bottom > tol
    
    print(f"Top :    {'TORN' if top_is_torn else 'STRAIGHT'} (variance: {std_top:.2f}px)")
    print(f"Bottom : {'TORN' if bottom_is_torn else 'STRAIGHT'} (variance: {std_bottom:.2f}px)")

    left_avg, right_avg = [], []
    for row in range(y_box, y_box + h_box):
        white_pixels = np.where(thresh[row, :] == 255)[0]
        if len(white_pixels) > 0:
            left_avg.append(int(white_pixels[0] - x_box))
            right_avg.append(int((x_box + w_box) - white_pixels[-1]))

    std_left = np.std(left_avg) if left_avg else 0
    std_right = np.std(right_avg) if right_avg else 0
    
    left_is_torn = std_left > tol
    right_is_torn = std_right > tol
    
    print(f"Left :   {'TORN' if left_is_torn else 'STRAIGHT'} (avg: {std_left:.2f}px)")
    print(f"Right :  {'TORN' if right_is_torn else 'STRAIGHT'} (avg: {std_right:.2f}px)")

    a1 = cv2.Canny(thresh, 150, 150)
    y_arr, x_arr = np.where(a1 == 255)
    
    y_pixels, x_pixels = np.where(thresh == 255)
    min_y, max_y = np.min(y_pixels), np.max(y_pixels)
    min_x, max_x = np.min(x_pixels), np.max(x_pixels)
    
    g1 = 5  
    any_torn = False

    if top_is_torn:
        any_torn = True
        b11_top = []
        for col_x in range(min_x, max_x + 1):
            pixels_in_col = y_arr[x_arr == col_x]
            if len(pixels_in_col) > 0:
                b11_top.append((int(col_x), int(np.min(pixels_in_col))))
        
        top_ref_y = int(min_y - g1)
        distances_top = [abs(int(y11) - top_ref_y) for _, y11 in b11_top]
        print(f"TOP Distances{distances_top}")
        topp.append(distances_top)

    if bottom_is_torn:
        any_torn = True
        b11_bottom = []
        for col_x in range(min_x, max_x + 1):
            pixels_in_col = y_arr[x_arr == col_x]
            if len(pixels_in_col) > 0:
                b11_bottom.append((int(col_x), int(np.max(pixels_in_col))))
        
        bottom_ref_y = int(max_y + g1)
        distances_bottom = [abs(int(y11) - bottom_ref_y) for _, y11 in b11_bottom]
        print(f"BOTTOM Distances{distances_bottom}")
        bottomm.append(distances_bottom)

    if left_is_torn:
        any_torn = True
        b11_left = []
        for row_y in range(min_y, max_y + 1):
            pixels_in_row = x_arr[y_arr == row_y]
            if len(pixels_in_row) > 0:
                b11_left.append((int(row_y), int(np.min(pixels_in_row))))
        
        left_ref_x = int(min_x - g1)
        distances_left = [abs(int(x11) - left_ref_x) for _, x11 in b11_left]
        print(f"LEFT Distances {distances_left}")
        leftt.append(distances_left)

    if right_is_torn:
        any_torn = True
        b11_right = []
        for row_y in range(min_y, max_y + 1):
            pixels_in_row = x_arr[y_arr == row_y]
            if len(pixels_in_row) > 0:
                b11_right.append((int(row_y), int(np.max(pixels_in_row))))
        
        right_ref_x = int(max_x + g1)
        distances_right = [abs(int(x11) - right_ref_x) for _, x11 in b11_right]
        print(f"RIGHT Distances{distances_right}")
        rightt.append(distances_right)
        

    

    plt.imshow(thresh, cmap='gray')
    plt.show()

    


In [ ]:
plt.imshow(isolated_pieces_list[4], cmap='gray')
plt.show()

In [ ]:
right_0=topp[0]
left_1=bottomm[4]
print(left_1)
print(right_0)

In [ ]:

shape1 = np.array(left_1)
shape2 = np.array(right_0)

if len(shape1) != len(shape2):
    min_len = min(len(shape1), len(shape2))
    shape1 = shape1[:min_len]
    shape2 = shape2[:min_len]

shape1_normalized = shape1 - np.mean(shape1)
shape2_normalized = shape2 - np.mean(shape2)
# print(shape1_normalized)
correlation = np.corrcoef(shape1_normalized, shape2_normalized)[0, 1]


print(correlation)

if correlation < -0.50 and correlation :
    print("match")
else:
    print("not")